In [1]:
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))

from config import config
from variables_naming import ARCHIVOS, VARIABLES

# Consolidar Tabla Final

**Asistido por Claude Opus 5.5**

Dado que la parte de modelamiento del paper se realiza sobre un solo año se decidio utilizar solo la ECV correspondiente al 2025 ya que es la mas reciente que se tiene hasta el momento

## Objetivo:

Generar un modelo capaz de clacificar si un hogar es multidimensionalmente pobre, por lo tanto el sujeto de estudio es el hogar y no el individuo.

In [2]:
CSV_DIR = config.EXTRACTED / "ECV_2025_extracted" / "csv"

KEY_COLS = ["DIRECTORIO", "SECUENCIA_ENCUESTA", "SECUENCIA_P", "ORDEN"]
SHORT = {
    "DIRECTORIO": "D",
    "SECUENCIA_ENCUESTA": "SE",
    "SECUENCIA_P": "SP",
    "ORDEN": "ORDEN",
}

# combinaciones que se usan a lo largo del notebook
D_SE = ["DIRECTORIO", "SECUENCIA_ENCUESTA"]
D_SP = ["DIRECTORIO", "SECUENCIA_P"]
D_SP_ORDEN = ["DIRECTORIO", "SECUENCIA_P", "ORDEN"]

# alias corto -> nombre del archivo, para no arrastrar tildes en el código
FILES = {
    "vivienda": "Datos de la vivienda.csv",
    "servicios": "Servicios del hogar.csv",
    "tenencia": (
        "Tenencia y financiacion de la vivienda que ocupa el hogar.csv"
    ),
    "condiciones": "Condiciones de vida del hogar y tenencia de bienes.csv",
    "adquisicion": "Adquisición de bienes y servicios.csv",
    "caracteristicas": "Características y composición del hogar.csv",
    "salud": "Salud.csv",
    "diseno": "Variables diseño muestral.csv",
    "educacion": "Educación.csv",
    "tic": "Tecnologías de información y comunicación.csv",
    "fuerza_trabajo": "Fuerza de trabajo.csv",
    "trabajo_infantil": "Trabajo infantil.csv",
    "menores_5": (
        "Atención integral de los niños y niñas menores de 5 años.csv"
    ),
    "item": "Adquisición de bienes y servicios por ítem.CSV",
    "programas": (
        "Condiciones de vida del hogar y tenencia de bienes (programas).csv"
    ),
    "escritura": (
        "Tenencia y financiacion de la vivienda que ocupa el hogar "
        "(personas escritura).csv"
    ),
    "pdet": "FEX_PDET_VIV_2025.csv",
}

# columnas auxiliares que se cargan junto con las llaves
EXTRA_COLS = {
    "servicios": ["CANT_PERSONAS_HOGAR"],
    "item": ["P3204"],
    "escritura": ["P8692S3A1"],
}

In [3]:
def read_data(alias, extra=()):
    """Lee las columnas llave presentes en el archivo, más `extra`."""
    path = CSV_DIR / FILES[alias]
    decimal = "." if alias == "item" else ","
    return pd.read_csv(path, sep=";", decimal=decimal)

In [4]:
keys = {alias: read_data(alias, EXTRA_COLS.get(alias, ())) for alias in FILES}

Una vez cargadas todas las tablas de manera individual se procede a consolidar las variables de interes, expuestas en la siguiente tabla, con su uso ya sea corte o modelamiento, y su nombre segun el diseño de la encuesta. Ademas antes de iniciar con el EDA y analisis previos al modelamiento se opta por realizar el Dual Cutoff Method (para crear la variable binaria objetivo) utilizando los cortes que el DANE reporta, adicionalmente se evita que variables que actuen en el corte se solapen con las predictoras. 

## Consolidar DF

La tabla final queda a **nivel persona**: una fila por persona con todas las
variables de la ECV 2025. Los cruces usan las llaves verificadas en
`full_analisis.ipynb`:

| Nivel | Llave en su archivo | Con qué se cruza en la tabla final |
| --- | --- | --- |
| vivienda | `DIRECTORIO` | `DIRECTORIO` |
| hogar | `DIRECTORIO` + `SECUENCIA_ENCUESTA` | `DIRECTORIO` + `SECUENCIA_P` |
| persona | `DIRECTORIO` + `SECUENCIA_P` + `ORDEN` | las mismas tres |

Tres detalles que salen de ese análisis:

- En los archivos de persona `SECUENCIA_ENCUESTA` repite lo que ya dice
  `ORDEN`, así que se descarta y la llave de persona queda `D + SP + ORDEN`.
- El hogar se llama `SECUENCIA_ENCUESTA` en los archivos de hogar y
  `SECUENCIA_P` en los de persona; por eso se renombra antes de cruzar.
- `FEX_C` es la única columna no llave que aparece en los 13 archivos. Se
  conserva solo la de `diseno`, porque `condiciones` y `diseno` difieren del
  resto en 4.553 hogares por redondeo.

Las variables de hogar y de vivienda **se repiten en cada integrante del
hogar**, que es lo que necesita el procesamiento posterior. `item`,
`programas` y `escritura` quedan fuera: traen varias filas por hogar y no
caben en una tabla de una fila por persona sin agregarlas antes.

In [5]:
# archivos que aportan una fila por persona y una fila por hogar
PERSON_FILES = [
    "salud",
    "diseno",
    "educacion",
    "tic",
    "fuerza_trabajo",
    "trabajo_infantil",
    "menores_5",
]
HOUSEHOLD_FILES = ["servicios", "tenencia", "condiciones", "adquisicion"]

In [6]:
# base: una fila por persona. SECUENCIA_ENCUESTA se descarta porque repite
# a ORDEN; FEX_C se conserva solo desde 'diseno'.
df = keys["caracteristicas"].drop(columns=["SECUENCIA_ENCUESTA", "FEX_C"])

for alias in PERSON_FILES:
    drop_cols = ["SECUENCIA_ENCUESTA"]
    if alias != "diseno":
        drop_cols.append("FEX_C")
    right = keys[alias].drop(columns=drop_cols, errors="ignore")
    df = df.merge(right, on=D_SP_ORDEN, how="left", validate="1:1")

print(df.shape)

(235350, 567)


In [7]:
# el hogar es SECUENCIA_ENCUESTA en estos archivos y SECUENCIA_P en la
# tabla de personas: se renombra para cruzar por D + SP
for alias in HOUSEHOLD_FILES:
    right = (
        keys[alias]
        .drop(columns=["SECUENCIA_P", "ORDEN", "FEX_C"])
        .rename(columns={"SECUENCIA_ENCUESTA": "SECUENCIA_P"})
    )
    df = df.merge(right, on=D_SP, how="left", validate="m:1")

print(df.shape)

(235350, 861)


In [8]:
# vivienda: sus SE, SP y ORDEN valen siempre 1, no aportan nada
right = keys["vivienda"].drop(
    columns=["SECUENCIA_ENCUESTA", "SECUENCIA_P", "ORDEN", "FEX_C"]
)
df = df.merge(right, on="DIRECTORIO", how="left", validate="m:1")

print(df.shape)

(235350, 902)


In [9]:
print(f"personas: {len(df)}")  # esperado 235350
print(
    f"iguales a 'caracteristicas': {len(df) == len(keys['caracteristicas'])}"
)
print(f"personas repetidas: {df.duplicated(subset=D_SP_ORDEN).sum()}")  # 0
print(
    f"columnas con sufijo: {[c for c in df.columns if c[-2:] in ('_x', '_y')]}"
)
print(f"hogares: {df[D_SP].drop_duplicates().shape[0]}")  # esperado 87060
print(f"viviendas: {df['DIRECTORIO'].nunique()}")  # esperado 86848

personas: 235350
iguales a 'caracteristicas': True
personas repetidas: 0
columnas con sufijo: []
hogares: 87060
viviendas: 86848


In [10]:
# los indicadores de hogar deben ser idénticos para todas las personas
# del hogar: un solo valor distinto por hogar
hogar_vars = ["CANT_PERSONAS_HOGAR", "PERCAPITA", "I_HOGAR", "P5000"]
print(df.groupby(D_SP)[hogar_vars].nunique().max())  # esperado 1 en todas

CANT_PERSONAS_HOGAR    1
PERCAPITA              1
I_HOGAR                1
P5000                  1
dtype: int64


1 significa: en el hogar más desviado de toda la encuesta, esa variable tiene un solo valor distinto. Es decir, no existe ni un hogar donde dos miembros tengan valores diferentes de un indicador de hogar

In [11]:
# la vivienda de dos hogares del contraejemplo de full_analisis.ipynb:
# CANT_PERSONAS_HOGAR se repite dentro del hogar, la edad cambia por persona
cols = [*D_SP_ORDEN, "CANT_PERSONAS_HOGAR", "P6040"]  # P6040 = edad
display(df[df["DIRECTORIO"] == 8370223][cols])

,DIRECTORIO,SECUENCIA_P,ORDEN,CANT_PERSONAS_HOGAR,P6040
1522,8370223,1,1,3,72
1523,8370223,1,2,3,45
1524,8370223,1,3,3,27
1525,8370223,2,1,2,48
1526,8370223,2,2,2,20


Totales de control: **235.350 personas**, 87.060 hogares y 86.848 viviendas,
los mismos del análisis de llaves.

Un `NaN` que venga de un módulo de persona (`educacion`, `tic`,
`fuerza_trabajo`, `trabajo_infantil`, `menores_5`) significa que **a esa
persona no le aplicaba el módulo** por edad, no que la respuesta falte:
`educacion` y `tic` cubren 221.043 personas, `fuerza_trabajo` 195.074,
`trabajo_infantil` 25.969 y `menores_5` 14.307.

El `validate=` de cada cruce es la red de seguridad: `1:1` en los módulos de
persona y `m:1` en hogar y vivienda, así que una llave equivocada levanta un
error en vez de multiplicar filas en silencio.

## Analisis general de tipos de datos

In [12]:
df.dtypes.value_counts()

float64    699
int64      202
str          1
Name: count, dtype: int64

In [13]:
cols_str = df.select_dtypes(include=["object", "string", "str"]).columns
print(cols_str)

Index(['BLOQUE'], dtype='str')


In [14]:
df["BLOQUE"]

0         A00013
1         A00013
2         A00013
3         A00013
4         A00013
           ...  
235345    R34C02
235346    R34C02
235347    R34C02
235348    R34C02
235349    R34C02
Name: BLOQUE, Length: 235350, dtype: str

In [15]:
for k, frame in keys.items():
    if "BLOQUE" in frame.columns:
        print(k)

diseno


`BLOQUE` es la única columna de texto de la tabla final y la única que no es
una respuesta de la encuesta: 

- **No es una predictora**, es una variable de diseño. Al armar la matriz del
  modelo hay que excluirla.
- `BLOQUE` es la partición geográfica en la que el DANE calibra los factores de expansión.
- Sirve para estimaciones por dominio (departamento, cabecera/resto) y, junto
  con `FEX_C`, `ESTRATO2020` y `SEGMENTO`, para calcular errores estándar
  respetando el diseño complejo de la encuesta.

### Usos posibles de `BLOQUE` (aportado por Claude Opus 5.5)

- Para estratificar el split. Separar train y test por BLOQUE y por el target mantiene la representatividad territorial, con SEGMENTO como grupo en el GroupKFold.
- Para evaluar el modelo. Mide el desempeño por bloque y así se ve si falla sistemáticamente en zonas rurales o en ciudades concretas.

In [16]:
df["BLOQUE"].nunique()

94

# Renombrar Columnas
**Asistido por claude Opus 5.5**

In [17]:
import re
import unicodedata

CODE_RE = re.compile(r"^P\d+[A-Za-z0-9]*$")
# F24 trae los sufijos en minúscula (P8587s1) y los CSV en mayúscula
V_UP = {k.upper(): v for k, v in VARIABLES.items()}


def _slug(texto):
    """Pasa la pregunta completa a snake_case sin tildes ni numeración."""
    t = re.sub(r"^\d+[A-Za-z]?\.\s*", "", texto)  # quita "4. ", "12A. "
    t = unicodedata.normalize("NFKD", t)
    t = t.encode("ascii", "ignore").decode()  # quita tildes
    return re.sub(r"[^0-9A-Za-z]+", "_", t).strip("_").lower()


def build_rename(columns):
    """código -> 'CÓDIGO_pregunta'; deja intacto lo que no es un P-código.

    El código se busca con `col.split("_")[0]`, así que la celda se puede
    volver a correr sobre un df que ya trae los nombres largos.
    """
    mapa = {}
    for col in columns:
        codigo = col.split("_")[0]
        info = V_UP.get(codigo.upper())
        if info and CODE_RE.match(codigo):
            pregunta = _slug(info["pregunta"])
            if pregunta:
                mapa[col] = f"{codigo}_{pregunta}"
    return mapa

In [18]:
rename_map = build_rename(df.columns)
df = df.rename(columns=rename_map)

print(f"renombradas: {len(rename_map)}")  # esperado 629
print(f"intactas: {df.shape[1] - len(rename_map)}")  # esperado 273
print(f"forma: {df.shape}")  # esperado (235350, 902)
print(f"nombres repetidos: {df.columns.duplicated().sum()}")  # esperado 0
print(f"llaves intactas: {all(c in df for c in [*D_SP_ORDEN, 'FEX_C'])}")
display(pd.Series(rename_map).head(8).to_frame("nombre nuevo"))

renombradas: 629
intactas: 273
forma: (235350, 902)
nombres repetidos: 0
llaves intactas: True


,nombre nuevo
P6016,P6016_numero_de_orden_de_la_persona_que_propor...
P1894,P1894_tipo_de_documento_de_identidad
P6020,P6020_sexo_al_nacer
P6034,P6034_cual_es_la_fecha_de_nacimiento_de
P6040,P6040_cuantos_anos_cumplidos_tiene
P6051,P6051_cual_es_el_parentesco_de_con_el_jefe_o_l...
P5502,P5502_actualmente_estado_civil
P6071,P6071_el_la_conyuge_de_vive_en_este_hogar


### Renombrar las columnas con `VARIABLES`

Cada columna que es un código ECV pasa a `CÓDIGO_pregunta`, con el texto
completo de la pregunta normalizado: sin tildes, en `snake_case` y sin la
numeración del cuestionario. Los nombres quedan largos (mediana 63
caracteres, máximo 111), pero no hay ambigüedad: en las familias de preguntas
se conserva tanto el encabezado común como la etiqueta que distingue
(`P1077S1` → `P1077S1_bienes_o_servicios_que_posee_el_hogar_lavadora_de_ropa`).

**El código se conserva al frente a propósito**: el IPM que falta especificar
en `variables_naming.py` indexa por código, y `col.split("_")[0]` lo recupera
sin necesidad de un diccionario inverso. Por eso mismo la celda se puede
volver a correr sobre un `df` ya renombrado.

**Qué queda intacto** (273 columnas):

- Las llaves (`DIRECTORIO`, `SECUENCIA_P`, `ORDEN`) y el bloque de diseño
  (`FEX_C`, `BLOQUE`, `SEGMENTO`, `ESTRATO2020`, `MPIO`, `F2`, `F_ETAPA1`,
  `FRES`), más `PERCAPITA`, `I_HOGAR`, `I_UGASTO`, `I_OU`,
  `CANT_PERSONAS_HOGAR`, `CLASE`, `REGION`, `P1_DEPARTAMENTO` y
  `P1_MUNICIPIO`: ya son nombres legibles y renombrarlos rompería `D_SP` y los
  chequeos de arriba.
- **251 columnas sin etiqueta en `VARIABLES`**, que siguen con su código
  pelado: `salud` 81, `adquisicion` 41, `caracteristicas` 33, `tic` 22,
  `menores_5` 22, `condiciones` 16, `servicios` 12, `educacion` 10,
  `fuerza_trabajo` 9, `vivienda` 4 y `trabajo_infantil` 1.

`adquisicion` no tiene ninguna etiqueta porque *"Adquisición de bienes y
servicios"* no está en `ARCHIVOS`; si esas 41 variables se van a usar, hay que
agregarlas al diccionario.

In [19]:
df

,DIRECTORIO,SECUENCIA_P,ORDEN,P6016_numero_de_orden_de_la_persona_que_proporciona_la_informacion,P1894_tipo_de_documento_de_identidad,P6020_sexo_al_nacer,P6034_cual_es_la_fecha_de_nacimiento_de,P6040_cuantos_anos_cumplidos_tiene,P6051_cual_es_el_parentesco_de_con_el_jefe_o_la_jefa_de_este_hogar,P5502_actualmente_estado_civil,...,P5661S9_frecuencia_de_problemas_en_el_sector_de_la_vivienda_contaminacion_en_rios_canales_lagos_y_embalses,P5661S5_frecuencia_de_problemas_en_el_sector_de_la_vivienda_invasion_del_espacio_publico,P5661S6_frecuencia_de_problemas_en_el_sector_de_la_vivienda_animales_que_causan_molestias,P5661S7_frecuencia_de_problemas_en_el_sector_de_la_vivienda_insectos_roedores,P3157,P3157S1_problemas_de_humedad_en_techos_o_paredes_pintura_soplada,P3157S2_problemas_de_humedad_en_techos_o_paredes_presencia_de_manchas,P3157S3_problemas_de_humedad_en_techos_o_paredes_presencia_de_moho,P3157S4_problemas_de_humedad_en_techos_o_paredes_cambios_en_el_aspecto_de_ladrillos_baldosas_concreto,P3158_los_techos_de_la_vivienda_presentan_goteras
0,8369550,1,1,1,3,2,1,45,1,5.0,...,1,1,1,1,NaN,2,2,2,2,2
1,8369550,1,2,2,3,1,1,25,7,4.0,...,1,1,1,1,NaN,2,2,2,2,2
2,8369550,1,3,3,2,2,1,16,9,5.0,...,1,1,1,1,NaN,2,2,2,2,2
3,8369550,1,4,4,2,1,1,10,9,5.0,...,1,1,1,1,NaN,2,2,2,2,2
4,8369551,1,1,1,3,2,1,62,1,3.0,...,1,1,2,1,NaN,2,2,2,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
235345,8489520,1,2,2,3,2,1,31,2,2.0,...,1,1,1,1,NaN,2,2,2,2,2
235346,8489520,1,3,3,1,2,1,8,3,NaN,...,1,1,1,1,NaN,2,2,2,2,2
235347,8489521,1,1,1,3,1,1,31,1,2.0,...,1,1,1,1,NaN,2,2,2,2,2
235348,8489521,1,2,2,3,2,1,32,2,2.0,...,1,1,1,1,NaN,2,2,2,2,2
